# Домашнее задание 1.3: Орбитальное движение в 2D

---

## Постановка физической задачи

Рассматривается движение материальной точки единичной массы в поле тяготения неподвижного притягивающего центра (гравитационная задача Кеплера в плоскости $Oxy$).

В безразмерных переменных (при $\mu = G M = 1$) уравнение движения имеет вид:
$$\ddot{\mathbf{r}} = -\frac{\mathbf{r}}{|\mathbf{r}|^3}, \qquad \mathbf{r} = (x, y)^T.$$

Перейдём к стандартной системе ОДУ 1-го порядка для вектора состояния $\mathbf{s} = (x, y, v_x, v_y)^T$:
$$\dot{\mathbf{r}} = \mathbf{v}, \qquad \dot{\mathbf{v}} = -\frac{\mathbf{r}}{r^3}, \quad r = \sqrt{x^2 + y^2}.$$

### Интегралы движения:
1. **Полная механическая энергия** (на единицу массы):
   $$E = \frac{v^2}{2} - \frac{1}{r} = \text{const}.$$
2. **Секториальная скорость / момент импульса** (на единицу массы):
   $$L_z = [\mathbf{r} \times \mathbf{v}]_z = x v_y - y v_x = \text{const}.$$

## 1. Численные методы интегрирования

Будем использовать два базовых метода численного интегрирования задачи Коши с постоянным шагом $h = \Delta t$:
1. **Явный метод Эйлера** (1-й порядок точности):
   $$\mathbf{s}_{n+1} = \mathbf{s}_n + h \mathbf{f}(\mathbf{s}_n).$$
2. **Классический метод Рунге-Кутты 4-го порядка (RK4)**:
   $$\mathbf{k}_1 = \mathbf{f}(\mathbf{s}_n),$$
   $$\mathbf{k}_2 = \mathbf{f}\left(\mathbf{s}_n + \frac{h}{2}\mathbf{k}_1\right),$$
   $$\mathbf{k}_3 = \mathbf{f}\left(\mathbf{s}_n + \frac{h}{2}\mathbf{k}_2\right),$$
   $$\mathbf{k}_4 = \mathbf{f}(\mathbf{s}_n + h\mathbf{k}_3),$$
   $$\mathbf{s}_{n+1} = \mathbf{s}_n + \frac{h}{6}(\mathbf{k}_1 + 2\mathbf{k}_2 + 2\mathbf{k}_3 + \mathbf{k}_4).$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon
from IPython.display import HTML, display

# Настройки оформления графиков
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3


def rhs(s):
    """
    Правая часть системы ОДУ для кеплеровского движения.
    s = [x, y, vx, vy]
    Возвращает [vx, vy, ax, ay].
    """
    x, y, vx, vy = s
    r = np.hypot(x, y)
    r3 = r**3
    ax = -x / r3
    ay = -y / r3
    return np.array([vx, vy, ax, ay])


def euler_step(s, dt):
    """Один шаг явного метода Эйлера."""
    return s + dt * rhs(s)


def rk4_step(s, dt):
    """Один шаг классического метода Рунге-Кутты 4-го порядка."""
    k1 = rhs(s)
    k2 = rhs(s + 0.5 * dt * k1)
    k3 = rhs(s + 0.5 * dt * k2)
    k4 = rhs(s + dt * k3)
    return s + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


def solve_ode(s0, dt, t_end, step_func=rk4_step):
    """
    Интегрирование системы от t=0 до t=t_end с фиксированным шагом dt.
    """
    n_steps = int(round(t_end / dt))
    t = np.linspace(0.0, t_end, n_steps + 1)
    states = np.empty((n_steps + 1, 4))
    states[0] = s0
    for i in range(n_steps):
        states[i + 1] = step_func(states[i], dt)
    return t, states


def calc_integrals(states):
    """
    Вычисление полной механической энергии E и момента импульса Lz вдоль траектории.
    """
    r = np.hypot(states[:, 0], states[:, 1])
    v2 = states[:, 2]**2 + states[:, 3]**2
    energy = 0.5 * v2 - 1.0 / r
    lz = states[:, 0] * states[:, 3] - states[:, 1] * states[:, 2]
    return energy, lz

---
## Часть 1: Тестирование на круговой орбите

В качестве эталонного теста берём круговую орбиту единичного радиуса с начальными условиями:
$$\mathbf{r}(0) = (1, 0), \qquad \mathbf{v}(0) = (0, 1).$$

Точное решение задачи Коши:
$$\mathbf{r}_{\text{exact}}(t) = (\cos t, \sin t), \qquad \mathbf{v}_{\text{exact}}(t) = (-\sin t, \cos t).$$
Период одного полного обращения: $T_0 = 2\pi$.  
Теоретические значения первых интегралов:
$$E_0 = \frac{1^2}{2} - \frac{1}{1} = -0.5, \qquad L_{z,0} = 1 \cdot 1 - 0 \cdot 0 = 1.0.$$

Выполним расчёт для нескольких шагов дискретизации: $h \in \{T_0/20, T_0/50, T_0/100, T_0/500\}$ на интервале в 5 полных периодов ($t \in [0, 5T_0]$).

In [ ]:
# Параметры круговой орбиты
s0_circle = np.array([1.0, 0.0, 0.0, 1.0])
T0 = 2.0 * np.pi
t_end_circle = 5.0 * T0

steps_circle = [T0 / 20.0, T0 / 50.0, T0 / 100.0, T0 / 500.0]

fig, axes = plt.subplots(2, 2, figsize=(11, 10))

# Точная форма круговой орбиты
phi_exact = np.linspace(0, 2 * np.pi, 300)
x_exact_circle = np.cos(phi_exact)
y_exact_circle = np.sin(phi_exact)

for ax, h in zip(axes.flat, steps_circle):
    t_e, s_e = solve_ode(s0_circle, h, t_end_circle, step_func=euler_step)
    t_r, s_r = solve_ode(s0_circle, h, t_end_circle, step_func=rk4_step)
    
    ax.plot(x_exact_circle, y_exact_circle, 'g-', lw=2, label='Точная')
    ax.plot(s_e[:, 0], s_e[:, 1], 'r--', lw=1.2, label='Эйлер')
    ax.plot(s_r[:, 0], s_r[:, 1], 'b-.', lw=1.2, label='RK4')
    ax.plot(s0_circle[0], s0_circle[1], 'ko', ms=5, label='Старт')
    
    n_per_T = int(round(T0 / h))
    ax.set_title(f'dt = T / {n_per_T}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=8)

fig.suptitle('Круговая орбита: сравнение Эйлера и RK4 с точным решением (5 оборотов)', fontsize=13)
fig.tight_layout()
plt.show()

### Сохранение энергии и момента импульса

Посмотрим на эволюцию интегралов движения во времени при шаге $h = T_0 / 200$.

* Метод Эйлера не является симплектическим и систематически «накачивает» энергию в систему: радиус орбиты монотонно растёт (раскручивающаяся спираль), полная энергия $E$ стремится к нулю и затем становится положительной (тело фиктивно преодолевает гравитационный захват). Момент импульса $L_z$ у явного метода Эйлера также испытывает численный дрейф.
* Метод RK4 сохраняет и энергию, и момент импульса с высочайшей точностью на рассматриваемых временах (отклонения не превышают порядка $10^{-7}$).

In [ ]:
h_test = T0 / 200.0
t_e, s_e = solve_ode(s0_circle, h_test, t_end_circle, step_func=euler_step)
t_r, s_r = solve_ode(s0_circle, h_test, t_end_circle, step_func=rk4_step)

E_e, L_e = calc_integrals(s_e)
E_r, L_r = calc_integrals(s_r)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Полная энергия
axes[0].plot(t_e, E_e, 'r--', lw=1.5, label='Эйлер')
axes[0].plot(t_r, E_r, 'b-', lw=1.5, label='RK4')
axes[0].axhline(-0.5, color='g', ls=':', lw=2, label='Точное (E = -0.5)')
axes[0].set_title('Полная механическая энергия E(t)')
axes[0].set_xlabel('Время t')
axes[0].set_ylabel('E')
axes[0].legend()

# Момент импульса
axes[1].plot(t_e, L_e, 'r--', lw=1.5, label='Эйлер')
axes[1].plot(t_r, L_r, 'b-', lw=1.5, label='RK4')
axes[1].axhline(1.0, color='g', ls=':', lw=2, label='Точное (Lz = 1.0)')
axes[1].set_title('Момент импульса Lz(t)')
axes[1].set_xlabel('Время t')
axes[1].set_ylabel('Lz')
axes[1].legend()

fig.suptitle(f'Эволюция первых интегралов движения (dt = T / 200)', fontsize=12)
fig.tight_layout()
plt.show()

---
## Часть 2: Эллиптические орбиты

### Аналитическое описание семейства орбит с одинаковой большой полуосью $a$

Пусть большая полуось зафиксирована: $a = 1.0$.
Рассмотрим семейство орбит с различными эксцентриситетами:
$$e \in \{0.0, 0.3, 0.6, 0.8\}.$$

Основные характеристики кеплеровского эллипса:
1. **Малая полуось**: $b = a \sqrt{1 - e^2}$.
2. **Расстояние до перицентра (перифокус)**: $r_p = a(1 - e)$.
3. **Расстояние до апоцентра (апофокус)**: $r_a = a(1 + e)$.
4. **Период обращения** (по 3-му закону Кеплера):
   $$T = 2\pi \sqrt{\frac{a^3}{\mu}} = 2\pi a^{3/2} = 2\pi \approx 6.2831853.$$
   *Замечание:* период зависит **исключительно от большой полуоси $a$** и абсолютно не зависит от эксцентриситета $e$! Все эллипсы с $a=1$ должны иметь строго один и тот же аналитический период $T = 2\pi$.
5. **Начальные условия в перицентре**:
   Поместим притягивающий центр в начало координат $(0, 0)$, а перицентр — на положительную полуось $x$.  
   Тогда $\mathbf{r}(0) = (r_p, 0) = (a(1-e), 0)$.  
   Скорость в перицентре направлена перпендикулярно радиус-вектору (вдоль оси $y$):
   $$v_p = \sqrt{\frac{1 + e}{a(1 - e)}}, \qquad \mathbf{v}(0) = (0, v_p).$$

In [ ]:
a = 1.0
eccentricities = [0.0, 0.3, 0.6, 0.8]
T_exact = 2.0 * np.pi * a**1.5

def get_initial_conditions(e, a=1.0):
    """Возвращает начальный вектор состояния в перицентре."""
    r_p = a * (1.0 - e)
    v_p = np.sqrt((1.0 + e) / (a * (1.0 - e)))
    return np.array([r_p, 0.0, 0.0, v_p])

print(f"Большая полуось: a = {a:.1f}")
print(f"Теоретический период всех орбит: T = {T_exact:.8f}")
print("-" * 68)
print(f"{'e':>5} | {'r_p (перицентр)':>15} | {'r_a (апоцентр)':>15} | {'v_p (скорость)':>15} | {'Энергия E':>10}")
print("-" * 68)
for e in eccentricities:
    s0_e = get_initial_conditions(e, a)
    E_0, _ = calc_integrals(s0_e.reshape(1, 4))
    r_p = a * (1 - e)
    r_a = a * (1 + e)
    v_p = s0_e[3]
    print(f"{e:5.1f} | {r_p:15.4f} | {r_a:15.4f} | {v_p:15.4f} | {E_0[0]:10.4f}")
print("-" * 68)

### Уравнение Кеплера и построение точной траектории

Для независимой проверки численного решения и сопоставления формы орбит нам потребуется точный закон движения во времени:
1. Средняя аномалия: $M(t) = n t = t / a^{3/2}$.
2. Эксцентрическая аномалия $E(t)$ удовлетворяет трансцендентному **уравнению Кеплера**:
   $$E - e \sin E = M.$$
   Решим его методом Ньютона (метод сходится за несколько итераций при $e < 1$).
3. Координаты тела в орбитальной плоскости:
   $$x(t) = a(\cos E(t) - e), \qquad y(t) = b\sin E(t) = a\sqrt{1 - e^2}\sin E(t).$$

In [5]:
def solve_kepler(M, e, tol=1e-12, max_iter=50):
    """Решение уравнения Кеплера E - e*sin(E) = M методом Ньютона."""
    E = M.copy()
    for _ in range(max_iter):
        f = E - e * np.sin(E) - M
        f_prime = 1.0 - e * np.cos(E)
        delta = -f / f_prime
        E += delta
        if np.max(np.abs(delta)) < tol:
            break
    return E


def exact_orbit_points(t_arr, e, a=1.0):
    """Вычисляет точные координаты (x, y) для моментов времени t_arr."""
    n = 1.0 / (a**1.5)
    M = n * t_arr
    E = solve_kepler(M, e)
    b = a * np.sqrt(1.0 - e**2)
    x = a * (np.cos(E) - e)
    y = b * np.sin(E)
    return x, y


def ellipse_parametric(e, a=1.0, num_pts=500):
    """Геометрическая линия эллипса для отрисовки фонового контура."""
    u = np.linspace(0, 2 * np.pi, num_pts)
    b = a * np.sqrt(1.0 - e**2)
    return a * (np.cos(u) - e), b * np.sin(u)

### Сравнение формы орбит и численное определение периодов

Проинтегрируем движение для каждого $e$ методом RK4 и измерим численный период обращения.

**Алгоритм измерения численного периода:**  
Определяем момент первого возвращения в положительную полуось $x$: момент смены знака координаты $y$ с отрицательного на положительный при $x > 0$ (пересечение перицентра снизу вверх). Для высокой точности (не ограниченной грубой сеткой $h$) применим линейную интерполяцию момента пересечения нуля внутри шага интегрирования.

Также оценим максимальную невязку формы полученной орбиты по каноническому уравнению эллипса:
$$q(x, y) = \frac{(x + ae)^2}{a^2} + \frac{y^2}{a^2(1 - e^2)} - 1.$$

In [ ]:
def measure_period(t, states):
    """
    Определение численного периода обращения по пересечению y=0 снизу вверх (vy > 0, x > 0).
    """
    y = states[:, 1]
    x = states[:, 0]
    
    # Ищем после первой трети оборота, чтобы не задеть начальный момент t=0
    start_idx = len(t) // 3
    crossings = np.where((y[start_idx:-1] < 0) & (y[start_idx+1:] >= 0) & (x[start_idx+1:] > 0))[0]
    if len(crossings) == 0:
        return np.nan
    idx = start_idx + crossings[0]
    
    y1, y2 = y[idx], y[idx+1]
    t1, t2 = t[idx], t[idx+1]
    t_cross = t1 + (-y1) / (y2 - y1) * (t2 - t1)
    return t_cross


# Сравнение формы орбит и измерение периода
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)

dt_sim = T_exact / 3000.0
t_sim_end = 1.2 * T_exact  # небольшой запас по времени для пересечения перицентра

print(f"{'e':>5} | {'Численный период T':>20} | {'Отн. ошибка периода':>22} | {'Макс. невязка формы':>22}")
print("-" * 78)

for ax, e in zip(axes.flat, eccentricities):
    s0_e = get_initial_conditions(e, a)
    t_num, s_num = solve_ode(s0_e, dt_sim, t_sim_end, step_func=rk4_step)
    
    T_meas = measure_period(t_num, s_num)
    rel_err_T = abs(T_meas - T_exact) / T_exact
    
    # Невязка формы эллипса за один период
    idx_1period = int(round(T_exact / dt_sim))
    x_orbit = s_num[:idx_1period + 1, 0]
    y_orbit = s_num[:idx_1period + 1, 1]
    b = a * np.sqrt(1 - e**2) if e < 1 else a
    residual = np.max(np.abs(((x_orbit + a*e)/a)**2 + (y_orbit/b)**2 - 1.0))
    
    print(f"{e:5.1f} | {T_meas:20.8f} | {rel_err_T:22.3e} | {residual:22.3e}")
    
    # Отрисовка
    x_cont, y_cont = ellipse_parametric(e, a)
    ax.plot(x_cont, y_cont, 'k--', lw=1, label='Аналитический эллипс')
    ax.plot(s_num[:, 0], s_num[:, 1], 'b-', lw=1.5, alpha=0.8, label='RK4 расчет')
    ax.plot(0, 0, 'r*', ms=8, label='Притягивающий центр')
    ax.plot(s0_e[0], s0_e[1], 'go', ms=5, label='Перицентр')
    
    ax.set_title(f'e = {e:.1f},  T_num = {T_meas:.4f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')
    ax.legend(loc='lower left', fontsize=7)

fig.suptitle('Форма эллиптических орбит при одинаковой большой полуоси a = 1.0', fontsize=12)
fig.tight_layout()
plt.show()

### Исследование ошибки возвращения через точный период $T$

Поскольку истинный период обращения $T = 2\pi a^{3/2}$ известен аналитически, оценим векторную ошибку возвращения тела в исходную точку перицентра ровно через один период:
$$\varepsilon_r(h) = \frac{\|\mathbf{r}_h(T) - \mathbf{r}_0\|}{a}.$$

Проведём расчёт для обоих методов (Эйлер и RK4) на серии измельчающихся сеток:
$$N \in \{250, 500, 1000, 2000, 4000\}, \qquad h = \frac{T}{N}.$$

In [ ]:
Ns = np.array([250, 500, 1000, 2000, 4000])

errors_rk4 = {e: [] for e in eccentricities}
errors_euler = {e: [] for e in eccentricities}

for e in eccentricities:
    s0_e = get_initial_conditions(e, a)
    for N in Ns:
        h = T_exact / N
        
        # RK4
        _, s_rk = solve_ode(s0_e, h, T_exact, step_func=rk4_step)
        err_r = np.linalg.norm(s_rk[-1, :2] - s0_e[:2]) / a
        errors_rk4[e].append(err_r)
        
        # Эйлер
        _, s_eu = solve_ode(s0_e, h, T_exact, step_func=euler_step)
        err_e = np.linalg.norm(s_eu[-1, :2] - s0_e[:2]) / a
        errors_euler[e].append(err_e)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# График для Эйлера
for e in eccentricities:
    axes[0].loglog(Ns, errors_euler[e], 'o--', label=f'e = {e:.1f}')
axes[0].set_title('Метод Эйлера: ошибка возвращения')
axes[0].set_xlabel('Число шагов N за период')
axes[0].set_ylabel('Ошибка ||r(T) - r0|| / a')
axes[0].legend()
axes[0].grid(True, which="both", ls="--")

# График для RK4
for e in eccentricities:
    axes[1].loglog(Ns, errors_rk4[e], 's-', label=f'e = {e:.1f}')
axes[1].set_title('Метод RK4: ошибка возвращения')
axes[1].set_xlabel('Число шагов N за период')
axes[1].set_ylabel('Ошибка ||r(T) - r0|| / a')
axes[1].legend()
axes[1].grid(True, which="both", ls="--")

fig.tight_layout()
plt.show()

# Печать таблицы результатов для RK4
print("Таблица ошибок возвращения для метода RK4:")
print(f"{'e':>5} | " + " | ".join([f'N={N:>4}' for N in Ns]) + " | Наблюдаемый порядок p")
print("-" * 75)
for e in eccentricities:
    err_list = errors_rk4[e]
    # Оценка наблюдаемого порядка сходимости p на двух последних сетках: log2(err[-2] / err[-1])
    p = np.log2(err_list[-2] / err_list[-1]) if err_list[-1] > 0 else 0
    row_str = " | ".join([f"{val:9.2e}" for val in err_list])
    print(f"{e:5.1f} | {row_str} | {p:10.2f}")
print("-" * 75)

### 
> **Как эксцентриситет влияет на распределение скорости вдоль орбиты и требования к шагу интегрирования?**

1. **Распределение скорости вдоль орбиты:**
   Из интеграла энергии $v^2 = \frac{2}{r} - \frac{1}{a}$ видно, что скорость сильно неоднородна:
   * В перицентре расстояние минимально $r_p = a(1 - e)$, а скорость максимальна:
     $$v_p = \sqrt{\frac{1 + e}{a(1 - e)}}.$$
   * В апоцентре расстояние максимально $r_a = a(1 + e)$, а скорость минимальна:
     $$v_a = \sqrt{\frac{1 - e}{a(1 + e)}}.$$
   * Отношение максимальной скорости к минимальной:
     $$\frac{v_p}{v_a} = \frac{1 + e}{1 - e}.$$
     * При $e = 0.0$ (круг): $v_p / v_a = 1.0$ (скорость постоянна).
     * При $e = 0.6$: $v_p / v_a = 1.6 / 0.4 = 4.0$.
     * При $e = 0.8$: $v_p / v_a = 1.8 / 0.2 = 9.0$!

2. **Влияние на требования к шагу интегрирования:**
   Вблизи перицентра гравитационное ускорение $|\ddot{\mathbf{r}}| = \frac{1}{r^2}$ возрастает в $\left(\frac{1+e}{1-e}\right)^2$ раз (при $e=0.8$ — в 81 раз по сравнению с апоцентром).  
   Характерное динамическое время пролёта окрестности перицентра можно оценить как:
   $$\tau_p \sim \frac{r_p}{v_p} = a^{3/2} \frac{(1 - e)^{3/2}}{\sqrt{1 + e}}.$$
   Для $e = 0.8$ масштаб $\tau_p \approx 0.067$, что в 15 раз меньше, чем для круговой орбиты $(\tau_p = 1.0)$.  
   **Вывод:** При использовании схемы с постоянным шагом $h$ величина шага диктуется самым жестким участком траектории — перицентром. Чем выше эксцентриситет $e$, тем меньший шаг $h$ требуется выбирать для сохранения заданной точности, иначе в перицентре накапливается грубая погрешность аппроксимации кривизны траектории.

In [ ]:
# График профиля скорости тела вдоль орбиты за один оборот
fig, ax = plt.subplots(figsize=(8.5, 4.2))

for e in eccentricities:
    s0_e = get_initial_conditions(e, a)
    t_orb, s_orb = solve_ode(s0_e, T_exact / 3000.0, T_exact, step_func=rk4_step)
    speed = np.hypot(s_orb[:, 2], s_orb[:, 3])
    ax.plot(t_orb / T_exact, speed, lw=1.8, label=f'e = {e:.1f}')

ax.set_title('Профиль скорости v(t) за один полный оборот')
ax.set_xlabel('Время t / T')
ax.set_ylabel('Скорость v')
ax.set_xlim(0, 1)
ax.legend(title='Эксцентриситет')
fig.tight_layout()
plt.show()

---
## Анимационные артефакты

### 1. Согласованная анимация методов Эйлера и RK4
Сравнение поведения численных схем при одинаковом шаге $h = T / 1000$ на вытянутой орбите $e = 0.6$.  
На обеих панелях пунктиром нанесена точная форма орбиты, а маркером показано текущее аналитическое положение (найденное из уравнения Кеплера).

In [ ]:
def create_methods_animation(e=0.6, N=1000, n_frames=100):
    """Создание анимации сравнения Эйлера и RK4 с точным решением."""
    h = T_exact / N
    s0 = get_initial_conditions(e, a)
    
    t_eu, s_eu = solve_ode(s0, h, T_exact, step_func=euler_step)
    t_rk, s_rk = solve_ode(s0, h, T_exact, step_func=rk4_step)
    
    # Индексы кадров для плавной анимации
    frame_indices = np.linspace(0, N, n_frames, dtype=int)
    t_frames = t_rk[frame_indices]
    
    # Точные аналитические координаты через уравнение Кеплера
    x_exact, y_exact = exact_orbit_points(t_frames, e, a)
    
    # Контур точной орбиты
    x_cont, y_cont = ellipse_parametric(e, a)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharex=True, sharey=True)
    
    # Лимиты осей
    xlim = (-1.8 * a, 0.8 * a)
    ylim = (-1.1 * a, 1.1 * a)
    
    lines = []
    points = []
    exact_points = []
    
    for ax, title, color in zip(axes, ['Метод Эйлера', 'Метод RK4'], ['tab:red', 'tab:blue']):
        ax.plot(x_cont, y_cont, 'k--', lw=1, alpha=0.6, label='Точная орбита')
        ax.plot(0, 0, 'y*', ms=10, mec='k', label='Центр')
        
        line, = ax.plot([], [], color=color, lw=1.2, label='Траектория')
        pt, = ax.plot([], [], 'o', color=color, ms=6)
        ex_pt, = ax.plot([], [], 'o', mfc='none', mec='green', mew=1.5, ms=8, label='Точное тело')
        
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_aspect('equal')
        ax.set_title(title)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.legend(loc='lower left', fontsize=8)
        
        lines.append(line)
        points.append(pt)
        exact_points.append(ex_pt)
    
    time_text = fig.suptitle(f'e = {e:.1f}, h = T/{N}, t/T = 0.00', fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    
    def update(frame_idx):
        idx = frame_indices[frame_idx]
        
        # Эйлер
        lines[0].set_data(s_eu[:idx+1, 0], s_eu[:idx+1, 1])
        points[0].set_data([s_eu[idx, 0]], [s_eu[idx, 1]])
        exact_points[0].set_data([x_exact[frame_idx]], [y_exact[frame_idx]])
        
        # RK4
        lines[1].set_data(s_rk[:idx+1, 0], s_rk[:idx+1, 1])
        points[1].set_data([s_rk[idx, 0]], [s_rk[idx, 1]])
        exact_points[1].set_data([x_exact[frame_idx]], [y_exact[frame_idx]])
        
        time_text.set_text(f'e = {e:.1f}, h = T/{N}, t/T = {t_rk[idx]/T_exact:.2f}')
        return lines + points + exact_points + [time_text]

    anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
    plt.close(fig)
    return anim

anim_methods = create_methods_animation(e=0.6, N=1000, n_frames=80)
HTML(anim_methods.to_jshtml())

### 2. Анимация ансамбля из 10 тел с разными эксцентриситетами
Все 10 тел имеют **одинаковую большую полуось $a = 1.0$**, но различные эксцентриситеты $e \in [0.0, 0.8]$.
Они одновременно стартуют из своих перицентров в момент $t = 0$.

* Все тела имеют абсолютно одинаковый период обращения $T = 2\pi$.
* Ровно в момент $t = T / 2$ все 10 тел одновременно оказываются в своих апоцентрах на отрицательной полуоси $x$.
* Ровно в момент $t = T$ все 10 тел одновременно собираются обратно в перицентрах.

In [ ]:
def create_ensemble_animation(n_bodies=10, n_frames=90):
    """Анимация одновременного движения 10 тел с разными e."""
    es = np.linspace(0.0, 0.8, n_bodies)
    colors = plt.cm.plasma(np.linspace(0.1, 0.9, n_bodies))
    
    # Интегрируем каждое тело с шагом RK4
    N = 2500
    h = T_exact / N
    frame_indices = np.linspace(0, N, n_frames, dtype=int)
    
    trajectories = []
    for e_val in es:
        s0_val = get_initial_conditions(e_val, a)
        _, s_val = solve_ode(s0_val, h, T_exact, step_func=rk4_step)
        trajectories.append(s_val)
    
    fig, ax = plt.subplots(figsize=(8, 5.5))
    ax.plot(0, 0, 'k*', ms=10, label='Центр притяжения')
    
    body_points = []
    for e_val, col in zip(es, colors):
        # Фоновый контур орбиты
        xc, yc = ellipse_parametric(e_val, a)
        ax.plot(xc, yc, color=col, lw=0.8, alpha=0.5)
        pt, = ax.plot([], [], 'o', color=col, ms=6, label=f'e={e_val:.2f}')
        body_points.append(pt)
    
    ax.set_xlim(-1.9 * a, 1.1 * a)
    ax.set_ylim(-1.1 * a, 1.1 * a)
    ax.set_aspect('equal')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    title = ax.set_title('Ансамбль орбит: t / T = 0.00')
    fig.tight_layout()
    
    def update(frame_idx):
        idx = frame_indices[frame_idx]
        for tr, pt in zip(trajectories, body_points):
            pt.set_data([tr[idx, 0]], [tr[idx, 1]])
        title.set_text(f'Ансамбль 10 орбит (a = {a:.1f}): t / T = {idx / N:.2f}')
        return body_points + [title]

    anim = FuncAnimation(fig, update, frames=n_frames, interval=45, blit=False)
    plt.close(fig)
    return anim

anim_ensemble = create_ensemble_animation(n_bodies=10, n_frames=90)
HTML(anim_ensemble.to_jshtml())

### 3. Закон равных площадей (Второй закон Кеплера)
Радиус-вектор тела заметает равные площади за равные промежутки времени:
$$\frac{dS}{dt} = \frac{1}{2}|\mathbf{r} \times \mathbf{v}| = \frac{L_z}{2} = \text{const}.$$

Разобьём период обращения по эллипсу $e = 0.6$ на $K = 12$ равных временных интервалов $\Delta t = T / 12$.  
Теоретическая площадь каждого сектора:
$$S_{\text{sector}} = \frac{\pi a b}{12} = \frac{\pi a^2 \sqrt{1 - e^2}}{12} \approx 0.2094395.$$

На согласованных панелях показаны:
* Слева: последовательное закрашивание секторов разными цветами по мере движения тела по орбите.
* Справа: график текущей скорости тела с бегущим маркером и цветовыми полосами, соответствующими каждому интервалу времени.

In [ ]:
def create_sectors_animation(e=0.6, n_sectors=12, n_frames=96):
    """Анимация 2-го закона Кеплера: закрашивание секторов и график скорости."""
    N = 2400
    h = T_exact / N
    s0 = get_initial_conditions(e, a)
    t_arr, s_arr = solve_ode(s0, h, T_exact, step_func=rk4_step)
    
    speed = np.hypot(s_arr[:, 2], s_arr[:, 3])
    steps_per_sec = N // n_sectors
    
    # Цветовая палитра для 12 секторов
    sector_colors = plt.cm.Set3(np.linspace(0, 1, n_sectors))
    
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    
    # Левая панель: орбита
    xc, yc = ellipse_parametric(e, a)
    axes[0].plot(xc, yc, 'k-', lw=1.2)
    axes[0].plot(0, 0, 'r*', ms=9, zorder=5)
    
    patches = []
    for sec_idx, col in enumerate(sector_colors):
        p = Polygon([[0, 0], [0, 0], [0, 0]], facecolor=col, alpha=0.6, edgecolor='none')
        axes[0].add_patch(p)
        patches.append(p)
        # Цветовые интервалы на правом графике скорости
        axes[1].axvspan(sec_idx / n_sectors, (sec_idx + 1) / n_sectors, color=col, alpha=0.3)
        
    body_marker, = axes[0].plot([], [], 'ko', ms=6, zorder=6)
    radius_line, = axes[0].plot([], [], 'k-', lw=1.2, zorder=4)
    axes[0].set_xlim(-1.8 * a, 0.6 * a)
    axes[0].set_ylim(-1.0 * a, 1.0 * a)
    axes[0].set_aspect('equal')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('y')
    axes[0].set_title('Заметаемые секторы орбиты (Δt = T/12)')
    
    # Правая панель: график скорости
    axes[1].plot(t_arr / T_exact, speed, 'k-', lw=1.5)
    cursor_line = axes[1].axvline(0, color='crimson', lw=1.5)
    speed_dot, = axes[1].plot([], [], 'o', color='crimson', ms=6)
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel('Время t / T')
    axes[1].set_ylabel('Скорость v')
    axes[1].set_title('Профиль скорости тела')
    
    frame_indices = np.linspace(0, N, n_frames, dtype=int)
    title = fig.suptitle(f'Второй закон Кеплера (e = {e:.1f}): t / T = 0.00', fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    
    def update(frame_idx):
        idx = frame_indices[frame_idx]
        
        # Обновление секторов
        for sec_idx, patch in enumerate(patches):
            start_i = sec_idx * steps_per_sec
            end_i = min(idx, (sec_idx + 1) * steps_per_sec)
            if end_i > start_i:
                pts = np.vstack([[0.0, 0.0], s_arr[start_i:end_i+1, :2], [0.0, 0.0]])
                patch.set_xy(pts)
                patch.set_visible(True)
            else:
                patch.set_visible(False)
                
        body_marker.set_data([s_arr[idx, 0]], [s_arr[idx, 1]])
        radius_line.set_data([0, s_arr[idx, 0]], [0, s_arr[idx, 1]])
        
        cursor_line.set_xdata([t_arr[idx] / T_exact, t_arr[idx] / T_exact])
        speed_dot.set_data([t_arr[idx] / T_exact], [speed[idx]])
        
        title.set_text(f'Второй закон Кеплера (e = {e:.1f}): t / T = {t_arr[idx] / T_exact:.2f}')
        return patches + [body_marker, radius_line, cursor_line, speed_dot, title]

    anim = FuncAnimation(fig, update, frames=n_frames, interval=45, blit=False)
    plt.close(fig)
    return anim

anim_sectors = create_sectors_animation(e=0.6, n_sectors=12, n_frames=96)
HTML(anim_sectors.to_jshtml())

---
## Выводы

1. **Анализ численных схем:**
   * Явный метод Эйлера 1-го порядка не пригоден для длительного моделирования кеплеровских орбит: не сохраняя симплектическую структуру задачи, он приводит к систематической накачке энергии в систему и раскрутке орбиты в расходящуюся спираль.
   * Метод Рунге-Кутты 4-го порядка (RK4) демонстрирует устойчивое сохранение энергии и момента импульса на временах порядка десятков периодов при умеренных шагах сетки, точно воспроизводя замкнутые орбиты.

2. **Эллиптические орбиты и законы Кеплера:**
   * Проверено теоретическое положение третьего закона Кеплера: при фиксированной большой полуоси $a$ период обращения строго постоянен и не зависит от эксцентриситета $e$.
   * Численно подтвержден второй закон Кеплера: равенство площадей секторов $\Delta S = \frac{1}{2} L_z \Delta t$, заметаемых радиус-вектором за равные интервалы времени.

3. **Влияние эксцентриситета на требования к шагу:**
   * С ростом эксцентриситета скорость в перицентре возрастает как $\sqrt{\frac{1+e}{1-e}}$, а ускорение — как $\frac{1}{(1-e)^2}$.
   * Для сохранения постоянной погрешности на орбитах с высоким эксцентриситетом при использовании равномерного шага по времени требуется существенно измельчать глобальный шаг сетки (ориентируясь на характерное динамическое время перицентра $\tau_p$), что наглядно обосновывает необходимость перехода к схемам с адаптивным шагом интегрирования или регуляризации времени (преобразование Сундмана).